# Lab 04 LSA Part 1 - Topic Modelling
In this lab we will look into building topic models, but will also examine dimensionality reduction and other relevant subjects.

## Latent Semantic Analysis (LSA)
Based on: [Latent Semantic Analysis using Python](https://www.datacamp.com/community/tutorials/discovering-hidden-topics-python)

In [1]:
from IPython.display import HTML, display
colab_button = HTML(
    '<a target="_blank" href="https://colab.research.google.com/github/surrey-nlp/NLP-2026/blob/main/lab04/lab04-topic-modelling-LSA.ipynb">'
    '<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>'
)
display(colab_button)

First lets add `data.zip` from https://github.com/surrey-nlp/NLP-2026/blob/main/lab04/data.zip to our working directory

In [2]:
!wget  https://raw.githubusercontent.com/surrey-nlp/NLP-2026/main/lab04/data.zip
!unzip ./data.zip

--2026-02-26 14:09:51--  https://raw.githubusercontent.com/surrey-nlp/NLP-2026/main/lab04/data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9646343 (9.2M) [application/zip]
Saving to: ‘data.zip’

data.zip            100%[===================>]   9.20M  --.-KB/s    in 0.03s   

2026-02-26 14:09:51 (346 MB/s) - ‘data.zip’ saved [9646343/9646343]

Archive:  ./data.zip
   creating: data/
  inflating: data/articles+4.txt     
  inflating: data/authors.csv        
  inflating: data/paper_authors.csv  


### Import the required library

In [3]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 62.9 MB/s eta 0:00:00


In [5]:
import os.path
import nltk
from gensim import corpora
from gensim.models import LsiModel
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from gensim.models.coherencemodel import CoherenceModel
import matplotlib.pyplot as plt
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

### Loading Data
This creates a data load function (for loading articles.csv later)

### Challenge 01

In [6]:
def load_data(path, file_name):
    """
    Input  : path and file_name
    Purpose: loading text file
    Output : list of paragraphs/documents and
             title(initial 100 words considered as title of document)
    """
    documents_list = []
    titles = []
    with open(os.path.join(path, file_name), "r") as fin:
        for line in fin.readlines():
            # TODO: strip (remove spaces at the start and end) a doc and add it to the documents list.
            text = line.strip()
            documents_list.append(text)

    print("Total Number of Documents:", len(documents_list))
    titles.append( text[0:min(len(text), 100)] )
    return documents_list,titles

### Preprocessing Data
After data loading function, you need to preprocess the text. Following steps are taken to preprocess the text:

Tokenize the text articles,
Remove stop words,
Perform stemming on text article.

### Challenge 02

In [7]:
def preprocess_data(doc_set):
    """
    Input  : document list
    Purpose: preprocess text (tokenize, removing stopwords, and stemming)
    Output : preprocessed text
    """
    # initialize regex tokenizer
    tokenizer = RegexpTokenizer(r'\w+')
    # create English stop words list
    en_stop = set(stopwords.words('english'))
    # Create p_stemmer of class PorterStemmer
    p_stemmer = PorterStemmer()
    # list for tokenized documents in loop
    texts = []
    # loop through document list
    for i in doc_set:
        # clean and tokenize document string

        # TODO: Convert into lower case and tokenize.
        raw = i.lower()
        tokens = tokenizer.tokenize(raw)

        # TODO: remove stop words from tokens
        stopped_tokens = [i for i in tokens if not i in en_stop]

        # TODO: stem tokens
        stemmed_tokens = [p_stemmer.stem(i) for i in stopped_tokens]

        # add tokens to list
        texts.append(stemmed_tokens)
    return texts

### Prepare Corpus
Next step is to prepare the corpus. Here, you need to create a document-term matrix and dictionary of terms.

In [ ]:
def prepare_corpus(doc_clean):
    """
    Input  : clean document
    Purpose: create term dictionary of our courpus and Converting list of documents (corpus) into Document Term Matrix
    Output : term dictionary and Document Term Matrix
    """
    # Creating the term dictionary of our courpus, where every unique term is assigned an index. dictionary = corpora.Dictionary(doc_clean)
    dictionary = corpora.Dictionary(doc_clean)

    # Converting list of documents (corpus) into Document Term Matrix using dictionary prepared above.
    doc_term_matrix = [dictionary.doc2bow(doc) for doc in doc_clean]

    return dictionary, doc_term_matrix

### Create an LSA model using Gensim
After corpus creation, you can generate a model using LSA.

In [ ]:
def create_gensim_lsa_model(doc_clean,number_of_topics,words):
    """
    Input  : clean document, number of topics and number of words associated with each topic
    Purpose: create LSA model using gensim
    Output : return LSA model
    """
    dictionary, doc_term_matrix = prepare_corpus(doc_clean)
    # generate LSA model
    lsamodel = LsiModel(doc_term_matrix, num_topics=number_of_topics, id2word = dictionary)  # train model
    print(lsamodel.print_topics(num_topics=number_of_topics, num_words=words))
    return lsamodel

### Determine the number of topics
Another extra step needs to be taken to optimize results by identifying an optimum amount of topics. Here, you will generate coherence scores to determine an optimum number of topics.

In [ ]:
def compute_coherence_values(dictionary, doc_term_matrix, doc_clean, stop, start=2, step=3):
    """
    Input   : dictionary : Gensim dictionary
              corpus : Gensim corpus
              texts : List of input texts
              stop : Max num of topics
    purpose : Compute c_v coherence for various number of topics
    Output  : model_list : List of LSA topic models
              coherence_values : Coherence values corresponding to the LDA model with respective number of topics
    """
    coherence_values = []
    model_list = []
    for num_topics in range(start, stop, step):
        # generate LSA model
        model = LsiModel(doc_term_matrix, num_topics=number_of_topics, id2word = dictionary)  # train model
        model_list.append(model)
        coherencemodel = CoherenceModel(model=model, texts=doc_clean, dictionary=dictionary, coherence='c_v')
        coherence_values.append(coherencemodel.get_coherence())
    return model_list, coherence_values

### Challenge 03
Let's plot coherence score values

In [ ]:
def plot_graph(doc_clean,start, stop, step):

    # TODO: Prepare corpus by calling prepare_corpus func.
    dictionary,doc_term_matrix = ...

    # TODO: Get coherence values by calling compute_coherence_values func.
    model_list, coherence_values = ...

    # Show graph
    x = range(start, stop, step)

    # TODO: Plot coherence score values
    plt.plot(...)

    plt.xlabel("Number of Topics")
    plt.ylabel("Coherence score")
    plt.legend(("coherence_values"), loc='best')
    plt.show()

You can easily evaluate this graph. Here, you have a number of topics on X-axis and coherence score on Y-axis. Of the number of topics, 7 have the highest coherence score, so the optimum number of topics are 7.

### Run it
Run all the above functions

In [ ]:
# LSA Model
number_of_topics=7
words=10
document_list,titles=load_data("", "./data/articles+4.txt")
clean_text=preprocess_data(document_list)
model=create_gensim_lsa_model(clean_text,number_of_topics,words)

In [ ]:
start,stop,step=2,12,1
plot_graph(clean_text,start,stop,step)